# RDD Analysis: Sharp Discontinuity with Diagnostics

This tutorial demonstrates Regression Discontinuity Design (RDD):
1. **Sharp RDD estimation** with local linear regression
2. **McCrary density test** for manipulation
3. **Bandwidth sensitivity analysis**
4. **Covariate balance** at the cutoff

**Scenario**: Effect of scholarship eligibility on college enrollment.
- Running variable: Test score (0-100)
- Cutoff: Score ≥ 70 qualifies for scholarship
- Outcome: College enrollment (binary or continuous)

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys
sys.path.insert(0, '../../src')

from causal_inference.rdd import sharp_rdd, mccrary_test, bandwidth_sensitivity

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## Generate RDD Data

We simulate test scores with a discontinuous jump in enrollment at the cutoff.

In [ ]:
# Parameters
n = 2000
cutoff = 70
true_effect = 0.15  # 15 percentage point increase in enrollment

# Generate running variable (test score)
# Use normal distribution centered around cutoff
X = np.random.normal(loc=cutoff, scale=15, size=n)
X = np.clip(X, 0, 100)  # Bound to [0, 100]

# Treatment indicator
T = (X >= cutoff).astype(int)

# Baseline enrollment probability (smooth function of score)
baseline = 0.3 + 0.005 * X + 0.0001 * (X - cutoff)**2

# Outcome with treatment effect at cutoff
prob = baseline + true_effect * T
prob = np.clip(prob, 0.05, 0.95)  # Bound probabilities
Y = np.random.binomial(1, prob)

# Create DataFrame
df = pd.DataFrame({
    'score': X,
    'enrolled': Y,
    'treated': T
})

print(f"Sample size: {n}")
print(f"Cutoff: {cutoff}")
print(f"True effect: {true_effect:.0%}")
print(f"N below cutoff: {sum(T == 0)} | N above: {sum(T == 1)}")
df.head()

## Visualize the Discontinuity

In [ ]:
# Binned means
df['score_bin'] = pd.cut(df['score'], bins=20)
binned = df.groupby('score_bin').agg({'score': 'mean', 'enrolled': 'mean'}).reset_index()

plt.figure(figsize=(10, 6))
plt.scatter(binned['score'], binned['enrolled'], s=50, alpha=0.8)
plt.axvline(x=cutoff, color='red', linestyle='--', label=f'Cutoff ({cutoff})')
plt.xlabel('Test Score')
plt.ylabel('Enrollment Rate')
plt.title('RDD: Enrollment by Test Score')
plt.legend()
plt.ylim(0, 1)
plt.show()

---

## Sharp RDD Estimation

Local linear regression estimates the discontinuity at the cutoff:

$$\hat{\tau}_{RDD} = \lim_{x \downarrow c} E[Y|X=x] - \lim_{x \uparrow c} E[Y|X=x]$$

In [ ]:
# Run Sharp RDD
result = sharp_rdd(
    Y=df['enrolled'].values,
    X=df['score'].values,
    cutoff=cutoff,
    bandwidth='optimal',  # IK or CCT optimal bandwidth
    kernel='triangular'
)

print("=== Sharp RDD Results ===")
print(f"Estimate: {result['estimate']:.3f}")
print(f"Std Error: {result['se']:.3f}")
print(f"95% CI: [{result['ci_lower']:.3f}, {result['ci_upper']:.3f}]")
print(f"p-value: {result['p_value']:.4f}")
print(f"\nBandwidth: {result['bandwidth']:.2f}")
print(f"Effective N (left): {result['n_left']}")
print(f"Effective N (right): {result['n_right']}")
print(f"\nTrue effect: {true_effect}")
print(f"Bias: {result['estimate'] - true_effect:.3f}")

In [ ]:
# Visualize RDD fit
h = result['bandwidth']

# Data within bandwidth
mask = np.abs(df['score'] - cutoff) <= h
df_bw = df[mask]

# Separate left/right
left = df_bw[df_bw['score'] < cutoff]
right = df_bw[df_bw['score'] >= cutoff]

# Fit lines
x_left = np.linspace(cutoff - h, cutoff, 50)
x_right = np.linspace(cutoff, cutoff + h, 50)

plt.figure(figsize=(10, 6))
plt.scatter(df['score'], df['enrolled'], alpha=0.3, s=10)
plt.axvline(x=cutoff, color='red', linestyle='--', label='Cutoff')
plt.axvline(x=cutoff - h, color='gray', linestyle=':', alpha=0.5)
plt.axvline(x=cutoff + h, color='gray', linestyle=':', alpha=0.5)

# Add estimated discontinuity
plt.plot([cutoff-0.5, cutoff-0.5], [result['left_intercept'], result['right_intercept']], 
         'g-', linewidth=3, label=f'Jump = {result["estimate"]:.3f}')

plt.xlabel('Test Score')
plt.ylabel('Enrollment')
plt.title(f'Sharp RDD with Bandwidth = {h:.1f}')
plt.legend()
plt.show()

---

## McCrary Density Test

**Purpose**: Test if units can precisely manipulate the running variable.

If students can manipulate their scores to just exceed the cutoff, we'd see a "bunching" of observations just above the cutoff. The McCrary test checks for a discontinuity in the density.

In [ ]:
# Run McCrary test
mccrary = mccrary_test(
    X=df['score'].values,
    cutoff=cutoff
)

print("=== McCrary Density Test ===")
print(f"Log difference in density: {mccrary['log_diff']:.3f}")
print(f"Std Error: {mccrary['se']:.3f}")
print(f"t-statistic: {mccrary['t_stat']:.3f}")
print(f"p-value: {mccrary['p_value']:.4f}")
print(f"\nInterpretation: {'PASS - No evidence of manipulation' if mccrary['p_value'] > 0.05 else 'FAIL - Evidence of manipulation'}")

In [ ]:
# Visualize density
fig, ax = plt.subplots(figsize=(10, 6))

# Histogram
bins = np.linspace(df['score'].min(), df['score'].max(), 40)
ax.hist(df['score'], bins=bins, density=True, alpha=0.7, edgecolor='white')
ax.axvline(x=cutoff, color='red', linestyle='--', label='Cutoff')
ax.set_xlabel('Test Score')
ax.set_ylabel('Density')
ax.set_title('McCrary Test: Running Variable Density')
ax.legend()
plt.show()

---

## Bandwidth Sensitivity Analysis

RDD estimates can be sensitive to bandwidth choice. We examine estimates across a range of bandwidths.

In [ ]:
# Bandwidth sensitivity
sensitivity = bandwidth_sensitivity(
    Y=df['enrolled'].values,
    X=df['score'].values,
    cutoff=cutoff,
    bandwidths=np.linspace(5, 25, 9)  # Range of bandwidths
)

print("=== Bandwidth Sensitivity ===")
print(f"{'Bandwidth':>10} {'Estimate':>10} {'SE':>10} {'N_eff':>10}")
print("-" * 45)
for bw, est, se, n_eff in zip(sensitivity['bandwidths'], 
                               sensitivity['estimates'],
                               sensitivity['ses'],
                               sensitivity['n_effective']):
    print(f"{bw:>10.1f} {est:>10.3f} {se:>10.3f} {n_eff:>10}")

In [ ]:
# Plot sensitivity
plt.figure(figsize=(10, 6))
plt.errorbar(sensitivity['bandwidths'], sensitivity['estimates'], 
             yerr=1.96*np.array(sensitivity['ses']), fmt='o-', capsize=3)
plt.axhline(y=true_effect, color='green', linestyle='--', label=f'True effect ({true_effect})')
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
plt.xlabel('Bandwidth')
plt.ylabel('RDD Estimate')
plt.title('Bandwidth Sensitivity Analysis')
plt.legend()
plt.show()

### Bandwidth Tradeoff

- **Smaller bandwidth**: Less bias (more local), but more variance (fewer observations)
- **Larger bandwidth**: More variance reduction, but potentially more bias from non-linearity
- **Optimal bandwidth**: Balances bias-variance tradeoff (IK or CCT)

---

## Covariate Balance at Cutoff

If the RDD is valid, covariates should be balanced at the cutoff (no discontinuity).

In [ ]:
# Add some covariates
df['age'] = 18 + np.random.normal(0, 1, n)
df['income'] = 50000 + 10000 * np.random.randn(n)
df['female'] = np.random.binomial(1, 0.5, n)

# Test balance for each covariate
print("=== Covariate Balance at Cutoff ===")
print(f"{'Covariate':>15} {'Estimate':>10} {'SE':>10} {'p-value':>10} {'Balance':>10}")
print("-" * 60)

for cov in ['age', 'income', 'female']:
    result_cov = sharp_rdd(
        Y=df[cov].values,
        X=df['score'].values,
        cutoff=cutoff,
        bandwidth='optimal'
    )
    balance = 'OK' if result_cov['p_value'] > 0.05 else 'FAIL'
    print(f"{cov:>15} {result_cov['estimate']:>10.3f} {result_cov['se']:>10.3f} {result_cov['p_value']:>10.3f} {balance:>10}")

---

## Summary: RDD Diagnostics Checklist

| Check | Purpose | Pass Criterion |
|-------|---------|---------------|
| **McCrary test** | No manipulation | p > 0.05 |
| **Bandwidth sensitivity** | Robust to bandwidth | Estimates stable |
| **Covariate balance** | No confounding | p > 0.05 for all |
| **Visual inspection** | Clear discontinuity | Jump visible at cutoff |

### Key Takeaways

1. **RDD identifies causal effects** when treatment is assigned by a cutoff
2. **Local validity**: Effect is identified only at the cutoff (not globally)
3. **McCrary test** is essential - if failed, RDD may be invalid
4. **Always check sensitivity** to bandwidth choice
5. **Covariate balance** supports the continuity assumption

---

*Created: Session 98 - Causal Inference Mastery*